# RentCheck AI — training

Runs one experiment, chosen by the `CONFIG` variable in the next cell.
`Runtime → Run all` does everything: repo, data, training, evaluation.

**One-time setup**

1. `Runtime → Change runtime type → GPU`
2. Left sidebar 🔑 **Secrets** → add `KAGGLE_USERNAME` and `KAGGLE_KEY`, enable *Notebook access* for both

**Safe to re-run.** Finished models are copied to `MyDrive/rentcheck-ai/models/` and
training is skipped if the model is already there, so `Run all` never retrains over
a finished result.

## Experiment

In [ ]:
CONFIG = 'baseline'          # baseline | ablation_noaug | main | res960 | arch_yolo11s | arch_yolo26s | size_yolov8m
BRANCH = 'training-pipeline'

import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB' if torch.cuda.is_available() else '-')
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU')

## Drive and repository

In [ ]:
from pathlib import Path
import subprocess
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/rentcheck-ai')
REPO = Path('/content/rentcheck-ai')
for sub in ('runs', 'models', 'results'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-f', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=False)
else:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH,
                    'https://github.com/nigarrustamova/rentcheck-ai.git', str(REPO)], check=True)

# Checkpoints live on Drive so a dropped session costs at most one epoch.
runs = REPO / 'runs'
if runs.exists() and not runs.is_symlink():
    import shutil
    shutil.rmtree(runs)
if not runs.exists():
    runs.symlink_to(DRIVE / 'runs')

print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

## Packages

In [ ]:
!pip install -q ultralytics==8.4.128 pycocotools

import ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

## Data

CarDD, downloaded from Kaggle to local disk (not Drive — Drive is far too slow for
thousands of small reads).

The CarDD licence forbids redistributing the dataset. This copy is for prototyping;
final numbers should use the copy obtained directly from the authors, and the paper
should cite that as the source.

In [ ]:
import os

RAW = Path('/content/data/raw')
CARDD = RAW / 'CarDD_release' / 'CarDD_COCO'

if not CARDD.exists():
    from google.colab import userdata
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:
        raise SystemExit(
            'Kaggle credentials missing. Left sidebar -> Secrets -> add '
            'KAGGLE_USERNAME and KAGGLE_KEY, enable Notebook access for both.'
        )
    RAW.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d nasimetemadi/car-damage-detection -p {RAW} --unzip -q

assert CARDD.exists(), f'Not found: {CARDD}'
print('data ready:', CARDD)

## Convert to YOLO format

In [ ]:
%cd /content/rentcheck-ai

PROCESSED = REPO / 'data' / 'processed' / 'cardd_seg' / 'dataset.yaml'
if PROCESSED.exists():
    print('already converted')
else:
    !python src/data/coco_to_yolo.py --src {CARDD} --dst data/processed

## Train

Skips entirely if this experiment already produced a model. Resumes from the last
checkpoint if a previous session was cut short.

In [ ]:
import yaml

cfg = yaml.safe_load((REPO / 'configs' / f'{CONFIG}.yaml').read_text())
NAME = cfg.get('name', CONFIG)
FINAL = DRIVE / 'models' / f'{NAME}.pt'
LAST = DRIVE / 'runs' / NAME / 'weights' / 'last.pt'

batch = cfg.get('batch', 16)
if torch.cuda.get_device_properties(0).total_memory / 1e9 < 15:
    batch = min(batch, 8)

print(f'config {CONFIG}  ->  run {NAME}   batch {batch}')

if FINAL.exists():
    print(f'Already trained: {FINAL}. Skipping. Delete that file to retrain.')
elif LAST.exists():
    print('Unfinished run found, resuming.')
    !python src/train/train.py --config configs/{CONFIG}.yaml --device 0 --batch {batch} --resume
else:
    !python src/train/train.py --config configs/{CONFIG}.yaml --device 0 --batch {batch}

## Evaluate and protect the result

Copies the weights out of `runs/` into `models/`, where no later training run can
reach them, then reports per-class AP on validation.

Only a *finished* run is copied. Ultralytics strips the optimizer out of the weights
when training completes, so that is what tells a finished run from an interrupted
one — and the training cell above skips any experiment that already has a model in
`models/`, which must not be true of a half-trained one.

In [ ]:
import shutil
import sys

sys.path.insert(0, str(REPO / 'src' / 'train'))
from train import is_resumable

best = DRIVE / 'runs' / NAME / 'weights' / 'best.pt'
last = DRIVE / 'runs' / NAME / 'weights' / 'last.pt'
finished = best.exists() and last.exists() and not is_resumable(last)

if not finished:
    print('Training has not finished yet — nothing copied. Re-run the cell above to continue.')
elif FINAL.exists():
    print(f'{FINAL.name} already saved.')
else:
    shutil.copy2(best, FINAL)
    print(f'saved {FINAL.name}  {FINAL.stat().st_size / 1e6:.1f} MB')

if FINAL.exists():
    !python src/eval/evaluate.py --weights {FINAL} --split val --device 0

    import pandas as pd
    csv = REPO / 'report' / 'results' / f'{NAME}_val.csv'
    if csv.exists():
        shutil.copy2(csv, DRIVE / 'results' / csv.name)
        display(pd.read_csv(csv).round(3))

## Next experiment

Change `CONFIG` in the second cell and run the notebook again. The ladder, each step
adding one thing to the step before it:

| Step | CONFIG | Pretrained | Augmentation | imgsz |
|---|---|---|---|---|
| 0 | `baseline` | no | no | 640 |
| 1 | `ablation_noaug` | yes | no | 640 |
| 2 | `main` | yes | yes | 640 |
| 3 | `res960` | yes | yes | 960 |

Side comparisons, all with the step 2 recipe: `arch_yolo11s`, `arch_yolo26s`, `size_yolov8m`.

**The test split stays untouched** until one model has been chosen on validation.